In [ ]:
import setting_for_sda.config as conf
from setting_for_sda.date_setting import Date_Setting
from setting_for_sda.path_setting import path_list
import lib.database.DBInterface as db_interface
import lib.utils.file_io as file_io

In [ ]:
# setting for the data extraction
lang='python'
year_start = 2021
year_end = 2025
schema = conf.database_info['schema']
year_range = f'{year_start}to{year_end}'

In [ ]:
monthly_timestamps = Date_Setting.year_range[year_range]["monthly_timestamps"]
save_dir = f"{path_list['data_root_dir']}/data/{schema}/questions/{lang}/{year_range}/"
file_io.create_dir(save_dir)
print("Data will be saved in ", save_dir)

In [ ]:
db_if = db_interface.DBInterface()

In [ ]:
for idx in range(len(monthly_timestamps)-1):
    st_dt   = monthly_timestamps[idx].replace('.','-')
    end_dt  = monthly_timestamps[idx+1].replace('.','-')

    # print(st_dt, end_dt)

    sql = """select p.id, p.creationdate, p.title, p.tags, p2.body from posts p , postsbody p2 where p.id = p2.id and p.posttypeid = '1' and p.tags like %s and p.creationdate >=  %s and p.creationdate < %s """
    rows = db_if.execute_query(sql, (f"%<{lang}>%", st_dt, end_dt))
   
    dict_q = [{ 'id' : row[0], 
                'creationdate' : row[1].isoformat(),
                'title' : row[2],
                'tags' : row[3],
                'body' : row[4]
            } for row in rows]
    
    print(f'size of the data from {st_dt} to {end_dt} : {len(dict_q)}')
    file_io.save_json(dict_q, f'{save_dir}{idx}.json')

